# 10년 배당 현금흐름 · 절세계좌 · 금융소득종합과세(2,000만 원) 시뮬레이터 (Google Colab)

**목적:** 단순한 미래 예측이 아니라, **지금 내 자산·월 적립금·A/B/C 배분 비중으로 앞으로 10~15년을 굴렸을 때**
1. **월 세후 배당금이 언제 목표 금액(월 100만 / 200만 / 300만 원)에 도달하는지**
2. **연간 배당소득이 금융소득종합과세 기준(연 2,000만 원)을 언제 돌파하는지** *(사전 절세 경보)*
3. **일반계좌 100% vs ISA·연금계좌 우선 배치 시 10년 누적 세금이 얼마나 차이 나는지**
4. **배분안(`50/20/30` vs `40/15/45` vs `국내배당 A 100%`)별로 보수적(하위 10%)·중립(50%)·낙관(상위 10%) 결과가 어떻게 달라지는지**
를 **API 키 없이** 상관관계 반영 몬테카를로 5,000회로 시뮬레이션하고, **초보자도 30초 만에 읽히는 5페이지 A4 PDF 리포트**를 Google Drive에 저장합니다.

> **개인 설정 안내:** 아래 **2번 셀의 `SIM_CONFIG`**에서 본인 초기 자산, 월 적립금, 계좌별 절세 비중만 바꾸면 바로 본인 맞춤 시뮬레이션이 실행됩니다. *(공개 저장소에는 어떠한 개인 계좌 금액도 포함되지 않습니다.)*


In [ ]:
# ==============================================================================
# 1. 환경 설정 · 한글 폰트(NanumGothic) 및 PDF 엔진 자동 설치 (API 키 불필요)
# ==============================================================================
import os, sys, io, math, base64, warnings
from pathlib import Path
from datetime import datetime, timezone, timedelta
from html import escape
import subprocess

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 및 PDF 엔진 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "libpango-1.0-0", "libpangoft2-1.0-0"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "weasyprint", "pyyaml"], check=False)

import numpy as np
import pandas as pd
import requests
import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

try:
    import yaml
except ImportError:
    yaml = None

try:
    from IPython.display import display
except ImportError:
    display = print

def setup_korean_font():
    chosen = None
    for p in [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothicBold.ttf",
    ]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()
    if chosen is None:
        avail = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
            if cand in avail:
                chosen = cand
                break
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

FONT_NAME = setup_korean_font()
print(f"[완료] 한글 폰트: {FONT_NAME} | Python {sys.version.split()[0]} | Colab={IN_COLAB}")

In [ ]:
# ==============================================================================
# 2. 내 시뮬레이션 조건 설정 · Google Drive 연결 (비공개 설정 자동 연동)
# ==============================================================================
SIM_CONFIG = {
    # ------------------------------------------------------------------
    # [본인 상황에 맞게 자유롭게 수정하세요 — 단위: 만 원]
    # ------------------------------------------------------------------
    "INIT_ASSET_MAN": 10000,          # 초기 투자 자산 (예: 10,000 = 1억 원)
    "MONTHLY_CONTRIB_MAN": 200,       # 매월 신규 적립금 (예: 200 = 월 200만 원)
    "HORIZON_YEARS": 12,              # 시뮬레이션 기간 (년, 기본 12년)
    "REINVEST_DIVIDEND": True,        # True: 배당금 전액 재투자(DRIP) / False: 배당금 생활비 인출

    # 목표 월 세후 배당금 (만 원 단위) — 도달 연도 계산용
    "TARGET_MONTHLY_DIVS_MAN": [100, 200, 300],

    # 세금 & 절세계좌 설정 (한국 세법 기준)
    "TAX_ADV_RATIO": 0.45,            # ISA·연금저축·IRP 등 절세계좌 비중 (예: 45%)
    "DIV_TAX_NORMAL": 0.154,          # 일반계좌 배당소득세 (15.4%)
    "DIV_TAX_ADV": 0.055,             # 절세계좌 실효세율 가정 (ISA 비과세/9.9%·연금저축 3.3~5.5% 혼합)
    "COMP_TAX_THRESHOLD_MAN": 2000,   # 금융소득종합과세 기준선 (일반계좌 연 2,000만 원)
    "COMP_TAX_EXTRA_RATE": 0.12,      # 2,000만 원 초과분 추가 누진세+건보료 부담 추정 (+12%p)

    # 비교할 3가지 자산배분 전략 (A: 국내 배당성장, B: 반도체·AI, C: 미국 지수)
    "STRATEGIES": {
        "안 ① 50/20/30 (현행 균형)":  {"A": 0.50, "B": 0.20, "C": 0.30},
        "안 ② 40/15/45 (미국 C 강화)": {"A": 0.40, "B": 0.15, "C": 0.45},
        "안 ③ A 100% (국내 배당 단독)": {"A": 1.00, "B": 0.00, "C": 0.00},
    },
    "PRIMARY_STRATEGY": "안 ① 50/20/30 (현행 균형)",

    # 몬테카를로 설정
    "N_SIMS": 5000,
    "RANDOM_SEED": 42,

    # Google Drive 저장 경로
    "USE_GOOGLE_DRIVE": True,
    "DRIVE_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_cashflow_sim",
    "PORTFOLIO_YAML": "/content/drive/MyDrive/Colab Notebooks/dividend_screener/my_kr_portfolio.yaml",
}

DRIVE_ROOT = None
DRIVE_REPORT_DIR = None
if IN_COLAB and SIM_CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(SIM_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 연결 완료] 리포트 저장 경로: {DRIVE_REPORT_DIR}")

        # 비공개 my_kr_portfolio.yaml에 basket_targets가 있으면 안 ① 목표 비중 자동 반영
        pf_p = Path(SIM_CONFIG["PORTFOLIO_YAML"])
        if yaml is not None and pf_p.exists():
            pf = yaml.safe_load(pf_p.read_text(encoding="utf-8")) or {}
            bt = pf.get("basket_targets") or {}
            if all(k in bt for k in ("A", "B", "C")):
                wA = float(bt["A"].get("target", 0.50))
                wB = float(bt["B"].get("target", 0.20))
                wC = float(bt["C"].get("target", 0.30))
                s = wA + wB + wC
                if s > 0:
                    SIM_CONFIG["STRATEGIES"]["안 ① 50/20/30 (현행 균형)"] = {"A": wA/s, "B": wB/s, "C": wC/s}
                    print(f"[비공개 YAML 연동] 목표 배분 A={wA/s:.0%} / B={wB/s:.0%} / C={wC/s:.0%} 확인 완료")
    except Exception as e:
        print(f"[Drive 미연결] 로컬 저장 후 Cell 5에서 재시도합니다 ({type(e).__name__}: {e})")

print(f"[설정 완료] 초기자산 {SIM_CONFIG['INIT_ASSET_MAN']:,}만 원 | 월 적립 {SIM_CONFIG['MONTHLY_CONTRIB_MAN']:,}만 원 | "
      f"기간 {SIM_CONFIG['HORIZON_YEARS']}년 | 절세계좌 비중 {SIM_CONFIG['TAX_ADV_RATIO']:.0%} | MC {SIM_CONFIG['N_SIMS']:,}회")

In [ ]:
# ==============================================================================
# 3. 실제 10년 시장 데이터 기반 층별(A·B·C) 수익률 · 변동성 · 배당률 · 상관관계 추정
# ==============================================================================
KST = timezone(timedelta(hours=9))
as_of_str = datetime.now(KST).strftime("%Y-%m-%d")
run_stamp = datetime.now(KST).strftime("%Y-%m-%d_%H%M%S")
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-cashflow-sim)"}

def fetch_yf_monthly_returns(symbol: str) -> pd.Series:
    try:
        r = requests.get(
            f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}",
            params={"range": "10y", "interval": "1mo"},
            headers=_HEADERS,
            timeout=15,
        )
        r.raise_for_status()
        res = r.json()["chart"]["result"][0]
        ts = res["timestamp"]
        adj = res["indicators"].get("adjclose", [{}])[0].get("adjclose")
        vals = adj if adj is not None else res["indicators"]["quote"][0]["close"]
        idx = [pd.Timestamp(datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m-01")) for t in ts]
        s = pd.Series(vals, index=idx, dtype=float).dropna()
        s = s[~s.index.duplicated(keep="last")].sort_index()
        if symbol == "KRW=X":
            s = s[(s >= 800.0) & (s <= 2000.0)]
        r_s = s.pct_change().dropna()
        return r_s[r_s.abs() < 0.60]
    except Exception:
        return pd.Series(dtype=float)

# 보수적·현실적 장기 가정 기본값 (한국 PIT 10년 백테스트 검증치 + 보수적 20% 헤어컷 반영)
# - A층(국내 배당성장): 연 총수익 9.5% (주가 +4.7% + 배당 4.8%, 배당성장 +5.5%/년), 연변동성 16.5%
# - B층(반도체·AI):    연 총수익 12.5% (주가 +10.5% + 배당 2.0%, 배당성장 +4.0%/년), 연변동성 28.0%
# - C층(미국 지수 원화): 연 총수익 10.5% (주가 +9.2% + 배당 1.3%, 배당성장 +6.0%/년), 연변동성 17.5%
LAYER_PARAMS = {
    "A": {"name": "A층 국내 배당성장", "mu_tr": 0.095, "vol": 0.165, "init_yld": 0.048, "dps_g": 0.055},
    "B": {"name": "B층 반도체·AI",    "mu_tr": 0.125, "vol": 0.280, "init_yld": 0.020, "dps_g": 0.040},
    "C": {"name": "C층 미국 지수(원화)", "mu_tr": 0.105, "vol": 0.175, "init_yld": 0.013, "dps_g": 0.060},
}

# 실제 Yahoo 10년 월봉으로 A·B·C 간 실측 상관관계 및 변동성 보정 시도
print("[시장 파라미터] Yahoo Finance 10년 월봉 기반 A·B·C 상관관계 실측 중...")
s_a_list = [fetch_yf_monthly_returns(sym) for sym in ("105560.KS", "000815.KS", "005389.KS")]
s_b = fetch_yf_monthly_returns("005935.KS")
s_spy = fetch_yf_monthly_returns("SPY")
s_qqq = fetch_yf_monthly_returns("QQQ")
s_fx = fetch_yf_monthly_returns("KRW=X")

CORR_MAT = np.array([
    [1.00, 0.48, 0.36],
    [0.48, 1.00, 0.52],
    [0.36, 0.52, 1.00],
], dtype=float)
CALIB_SOURCE = "장기 보수적 기준치 (오프라인 폴백)"

valid_a = [s for s in s_a_list if len(s) >= 60]
if valid_a and len(s_b) >= 60 and len(s_spy) >= 60:
    df_a = pd.concat(valid_a, axis=1).mean(axis=1)
    df_c_usd = (0.5 * s_spy + 0.5 * s_qqq).dropna() if len(s_qqq) >= 60 else s_spy
    if len(s_fx) >= 60:
        df_c = ((1.0 + df_c_usd) * (1.0 + s_fx) - 1.0).dropna()
    else:
        df_c = df_c_usd
    comb = pd.concat({"A": df_a, "B": s_b, "C": df_c}, axis=1).dropna()
    if len(comb) >= 48:
        CORR_MAT = comb.corr().values
        # 실측 변동성을 상·하한 캡 안에서 반영 (극단 왜곡 방지)
        obs_vol = comb.std() * math.sqrt(12)
        LAYER_PARAMS["A"]["vol"] = float(np.clip(obs_vol["A"], 0.14, 0.24))
        LAYER_PARAMS["B"]["vol"] = float(np.clip(obs_vol["B"], 0.22, 0.34))
        LAYER_PARAMS["C"]["vol"] = float(np.clip(obs_vol["C"], 0.14, 0.23))
        CALIB_SOURCE = f"Yahoo 10년 실측 상관·변동성({len(comb)}개월) + 보수적 기대수익률"

print(f"[보정 완료] {CALIB_SOURCE}")
for k, v in LAYER_PARAMS.items():
    print(f"  - {v['name']}: 연 기대총수익 {v['mu_tr']:.1%} | 연 변동성 {v['vol']:.1%} | 현재 배당률 {v['init_yld']:.1%} | 배당성장 {v['dps_g']:.1%}/년")
print(f"  - 층간 상관계수: A-B={CORR_MAT[0,1]:.2f}, A-C={CORR_MAT[0,2]:.2f}, B-C={CORR_MAT[1,2]:.2f}")

In [ ]:
# ==============================================================================
# 4. 5,000회 상관관계 몬테카를로 시뮬레이션 · 절세효과 · 목표 월배당 도달 연도 분석
# ==============================================================================
def run_monte_carlo(weights: dict, init_man: float, monthly_man: float, years: int,
                    tax_adv_ratio: float, reinvest: bool = True, n_sims: int = 5000, seed: int = 42) -> dict:
    rng = np.random.default_rng(seed)
    w = np.array([weights["A"], weights["B"], weights["C"]], dtype=float)
    w = w / w.sum()

    # 층별 연간 주가수익률(=총수익률-배당률)과 변동성
    mu_tr = np.array([LAYER_PARAMS[k]["mu_tr"] for k in ("A", "B", "C")])
    yld0  = np.array([LAYER_PARAMS[k]["init_yld"] for k in ("A", "B", "C")])
    dps_g = np.array([LAYER_PARAMS[k]["dps_g"] for k in ("A", "B", "C")])
    vols  = np.array([LAYER_PARAMS[k]["vol"] for k in ("A", "B", "C")])

    # 월간 로그수익률 평균/공분산 (주가 수익률 성분)
    mu_price_yr = mu_tr - yld0
    cov_yr = np.outer(vols, vols) * CORR_MAT
    cov_m = cov_yr / 12.0
    L = np.linalg.cholesky(cov_m + np.eye(3) * 1e-9)
    drift_m = (mu_price_yr - 0.5 * (vols ** 2)) / 12.0

    # 포트폴리오 초기 가중 배당률 & 배당성장률
    port_yld0 = float(np.dot(w, yld0))
    port_dps_g = float(np.dot(w * yld0, dps_g) / max(port_yld0, 1e-6))

    months = years * 12
    asset_paths = np.zeros((n_sims, years + 1), dtype=float)
    gross_div_yr = np.zeros((n_sims, years + 1), dtype=float)
    net_div_yr = np.zeros((n_sims, years + 1), dtype=float)
    tax_paid_yr = np.zeros((n_sims, years + 1), dtype=float)
    comp_tax_hit = np.zeros((n_sims, years + 1), dtype=bool)

    # 현재 0년차 기준값
    asset_paths[:, 0] = init_man
    g0 = init_man * port_yld0
    gross_div_yr[:, 0] = g0
    normal_g0 = g0 * (1.0 - tax_adv_ratio)
    adv_g0 = g0 * tax_adv_ratio
    extra0 = max(0.0, normal_g0 - SIM_CONFIG["COMP_TAX_THRESHOLD_MAN"]) * SIM_CONFIG["COMP_TAX_EXTRA_RATE"]
    t0 = normal_g0 * SIM_CONFIG["DIV_TAX_NORMAL"] + adv_g0 * SIM_CONFIG["DIV_TAX_ADV"] + extra0
    tax_paid_yr[:, 0] = t0
    net_div_yr[:, 0] = g0 - t0
    comp_tax_hit[:, 0] = normal_g0 >= SIM_CONFIG["COMP_TAX_THRESHOLD_MAN"]

    # 시뮬레이션 루프 (연 단위 리밸런싱 + 월 단위 주가·적립 반영)
    cur_asset = np.full(n_sims, float(init_man), dtype=float)
    # 배당수익률은 원가·시장 평균 회귀를 반영해 안정적 현실성 확보
    for yr in range(1, years + 1):
        yr_gross_div = np.zeros(n_sims, dtype=float)
        # 층별 배당률은 주가 변동에 따라 완만히 연동 (과도한 배당률 폭증 방지)
        for m in range(12):
            z = rng.standard_normal((n_sims, 3))
            r_m = np.exp(drift_m + z @ L.T) - 1.0
            port_r_m = r_m @ w
            cur_asset = np.maximum(cur_asset * (1.0 + port_r_m) + monthly_man, 1.0)
            # 월 배당 발생액 (연간 배당률의 1/12 + 배당성장률 완만 반영, 상한 캡 적용)
            eff_yld = min(port_yld0 * ((1.0 + 0.15 * port_dps_g) ** yr), port_yld0 * 1.35)
            m_div = cur_asset * (eff_yld / 12.0)
            yr_gross_div += m_div
            if reinvest:
                # 월별 세후 배당금 자동 재투자
                m_norm = m_div * (1.0 - tax_adv_ratio)
                m_adv  = m_div * tax_adv_ratio
                m_net  = m_norm * (1.0 - SIM_CONFIG["DIV_TAX_NORMAL"]) + m_adv * (1.0 - SIM_CONFIG["DIV_TAX_ADV"])
                cur_asset += m_net

        # 연간 세금 정산 (일반계좌 2,000만 원 초과 시 금융소득종합과세 추가 부담 반영)
        norm_div = yr_gross_div * (1.0 - tax_adv_ratio)
        adv_div  = yr_gross_div * tax_adv_ratio
        base_tax = norm_div * SIM_CONFIG["DIV_TAX_NORMAL"] + adv_div * SIM_CONFIG["DIV_TAX_ADV"]
        extra_tax = np.maximum(0.0, norm_div - SIM_CONFIG["COMP_TAX_THRESHOLD_MAN"]) * SIM_CONFIG["COMP_TAX_EXTRA_RATE"]
        tot_tax = base_tax + extra_tax
        if reinvest:
            cur_asset = np.maximum(cur_asset - extra_tax, 1.0)

        asset_paths[:, yr] = cur_asset
        gross_div_yr[:, yr] = yr_gross_div
        tax_paid_yr[:, yr] = tot_tax
        net_div_yr[:, yr] = yr_gross_div - tot_tax
        comp_tax_hit[:, yr] = norm_div >= SIM_CONFIG["COMP_TAX_THRESHOLD_MAN"]

    return {
        "weights": weights,
        "port_yld0": port_yld0,
        "asset_p10": np.percentile(asset_paths, 10, axis=0),
        "asset_p50": np.percentile(asset_paths, 50, axis=0),
        "asset_p90": np.percentile(asset_paths, 90, axis=0),
        "mdiv_p10": np.percentile(net_div_yr / 12.0, 10, axis=0),
        "mdiv_p50": np.percentile(net_div_yr / 12.0, 50, axis=0),
        "mdiv_p90": np.percentile(net_div_yr / 12.0, 90, axis=0),
        "norm_div_p50": np.percentile(gross_div_yr * (1.0 - tax_adv_ratio), 50, axis=0),
        "gross_div_p50": np.percentile(gross_div_yr, 50, axis=0),
        "cum_tax_p50": np.percentile(np.cumsum(tax_paid_yr[:, 1:], axis=1), 50, axis=0),
        "comp_hit_prob": comp_tax_hit.mean(axis=0) * 100.0,
    }

H = SIM_CONFIG["HORIZON_YEARS"]
YEARS_AXIS = np.arange(0, H + 1)
PRINCIPAL_AXIS = SIM_CONFIG["INIT_ASSET_MAN"] + SIM_CONFIG["MONTHLY_CONTRIB_MAN"] * 12 * YEARS_AXIS

# 1) 3대 배분 전략 비교 (절세계좌 설정 반영)
STRAT_RES = {}
for sname, sw in SIM_CONFIG["STRATEGIES"].items():
    STRAT_RES[sname] = run_monte_carlo(
        sw, SIM_CONFIG["INIT_ASSET_MAN"], SIM_CONFIG["MONTHLY_CONTRIB_MAN"],
        H, SIM_CONFIG["TAX_ADV_RATIO"], SIM_CONFIG["REINVEST_DIVIDEND"],
        SIM_CONFIG["N_SIMS"], SIM_CONFIG["RANDOM_SEED"],
    )

PRIMARY_NAME = SIM_CONFIG["PRIMARY_STRATEGY"]
PRI_RES = STRAT_RES[PRIMARY_NAME]

# 2) 절세계좌 효과 비교 (일반계좌 100% vs 설정 절세비중 vs 절세 70%)
TAX_COMP = {
    "일반계좌 100% (절세 0%)": run_monte_carlo(
        SIM_CONFIG["STRATEGIES"][PRIMARY_NAME], SIM_CONFIG["INIT_ASSET_MAN"], SIM_CONFIG["MONTHLY_CONTRIB_MAN"],
        H, 0.0, SIM_CONFIG["REINVEST_DIVIDEND"], SIM_CONFIG["N_SIMS"], SIM_CONFIG["RANDOM_SEED"],
    ),
    f"현행 절세배치 ({SIM_CONFIG['TAX_ADV_RATIO']:.0%})": PRI_RES,
    "절세 극대화 (70%)": run_monte_carlo(
        SIM_CONFIG["STRATEGIES"][PRIMARY_NAME], SIM_CONFIG["INIT_ASSET_MAN"], SIM_CONFIG["MONTHLY_CONTRIB_MAN"],
        H, 0.70, SIM_CONFIG["REINVEST_DIVIDEND"], SIM_CONFIG["N_SIMS"], SIM_CONFIG["RANDOM_SEED"],
    ),
}

# 3) 월 적립금 민감도 분석 (목표 월배당 100 / 200 / 300만 원 도달 연도)
CONTRIB_GRID = sorted({max(50, int(SIM_CONFIG["MONTHLY_CONTRIB_MAN"] * f)) for f in (0.5, 1.0, 1.5, 2.0)})
GOAL_ROWS = []
for mc_man in CONTRIB_GRID:
    res_g = run_monte_carlo(
        SIM_CONFIG["STRATEGIES"][PRIMARY_NAME], SIM_CONFIG["INIT_ASSET_MAN"], mc_man,
        15, SIM_CONFIG["TAX_ADV_RATIO"], SIM_CONFIG["REINVEST_DIVIDEND"], 2000, SIM_CONFIG["RANDOM_SEED"],
    )
    row = {"월 적립금": f"{mc_man:,}만 원", "10년차 중앙자산": f"{res_g['asset_p50'][10]/10000:.2f}억", "10년차 세후 월배당": f"{res_g['mdiv_p50'][10]:.0f}만 원"}
    for tgt in SIM_CONFIG["TARGET_MONTHLY_DIVS_MAN"]:
        hits = np.where(res_g["mdiv_p50"] >= tgt)[0]
        row[f"월 {tgt}만 달성"] = f"{int(hits[0])}년차" if len(hits) else "15년 초과"
    GOAL_ROWS.append(row)
GOAL_DF = pd.DataFrame(GOAL_ROWS)

# 요약 테이블
SUMMARY_ROWS = []
for sname, res in STRAT_RES.items():
    y10 = min(10, H)
    comp_yrs = np.where(res["norm_div_p50"] >= SIM_CONFIG["COMP_TAX_THRESHOLD_MAN"])[0]
    comp_str = f"{int(comp_yrs[0])}년차 경보" if len(comp_yrs) else f"{H}년 내 안전"
    SUMMARY_ROWS.append({
        "배분 전략": sname,
        "초기 배당률": f"{res['port_yld0']:.2%}",
        f"{y10}년차 자산(하위10%)": f"{res['asset_p10'][y10]/10000:.2f}억",
        f"{y10}년차 자산(중앙50%)": f"{res['asset_p50'][y10]/10000:.2f}억",
        f"{y10}년차 자산(상위10%)": f"{res['asset_p90'][y10]/10000:.2f}억",
        f"{y10}년차 세후 월배당": f"{res['mdiv_p50'][y10]:.0f}만 원",
        f"{y10}년 누적 세금": f"{res['cum_tax_p50'][y10-1]:,.0f}만 원",
        "종소세(2천만) 돌파": comp_str,
    })
SUMMARY_SIM_DF = pd.DataFrame(SUMMARY_ROWS)

print(f"■ [10년 배당 현금흐름 · 절세 시뮬레이션 요약 ({as_of_str})]")
display(SUMMARY_SIM_DF)
print("\n■ [월 적립금별 목표 세후 월배당 달성 연도 (중앙값 기준)]")
display(GOAL_DF)

In [ ]:
# ==============================================================================
# 5. 고가독성 풀폭 차트 4종 + 5페이지 A4 PDF/HTML 리포트 생성 & Drive 자동 저장
# ==============================================================================
def export_html_to_pdf(html_str: str, pdf_path: str, chart_b64_list: list[str]) -> bool:
    try:
        from weasyprint import HTML as WP_HTML
        WP_HTML(string=html_str).write_pdf(pdf_path)
        return True
    except Exception as e:
        print(f"[안내] WeasyPrint 폴백({type(e).__name__}: {e}) -> Matplotlib PDF로 생성")
    try:
        from matplotlib.backends.backend_pdf import PdfPages
        import matplotlib.image as mpimg
        with PdfPages(pdf_path) as pdf:
            for b64 in chart_b64_list:
                img = mpimg.imread(io.BytesIO(base64.b64decode(b64)), format="png")
                fig_p = plt.figure(figsize=(11.69, 8.27))
                plt.imshow(img); plt.axis("off"); plt.tight_layout(); pdf.savefig(fig_p, dpi=150); plt.close(fig_p)
        return True
    except Exception:
        return False

y10 = min(10, H)

# ------------------------------------------------------------------------------
# 차트 ① (풀폭 단독): 10~12년 자산 성장 시나리오 팬 차트 (하위10% · 중앙50% · 상위10%)
# ------------------------------------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(13.8, 5.8), facecolor="white")
ax1.fill_between(YEARS_AXIS, PRI_RES["asset_p10"]/10000, PRI_RES["asset_p90"]/10000,
                 color="#99F6E4", alpha=0.40, label="전체 80% 가능 범위 (보수 하위10% ~ 낙관 상위10%)")
ax1.plot(YEARS_AXIS, PRI_RES["asset_p90"]/10000, color="#059669", ls="--", lw=1.6, label=f"낙관 시나리오 (상위 10% · {H}년차 {PRI_RES['asset_p90'][-1]/10000:.1f}억)")
ax1.plot(YEARS_AXIS, PRI_RES["asset_p50"]/10000, color="#0F2922", lw=3.0, marker="o", ms=5, label=f"기본 시나리오 (중앙값 50% · {H}년차 {PRI_RES['asset_p50'][-1]/10000:.1f}억)")
ax1.plot(YEARS_AXIS, PRI_RES["asset_p10"]/10000, color="#D97706", ls="--", lw=1.8, label=f"보수 시나리오 (하위 10% · {H}년차 {PRI_RES['asset_p10'][-1]/10000:.1f}억)")
ax1.plot(YEARS_AXIS, PRINCIPAL_AXIS/10000, color="#64748B", ls=":", lw=2.0, label=f"내 누적 투자원금 ({H}년차 {PRINCIPAL_AXIS[-1]/10000:.1f}억)")

for yr_mark in sorted({min(5, H), y10, H}):
    val_m = PRI_RES["asset_p50"][yr_mark] / 10000
    ax1.annotate(f"{yr_mark}년차\n{val_m:.2f}억", xy=(yr_mark, val_m), xytext=(0, 14), textcoords="offset points",
                 ha="center", fontsize=9.0, fontweight="bold", color="#0F2922",
                 bbox=dict(boxstyle="round,pad=0.22", facecolor="white", edgecolor="#0F766E", lw=0.9))

ax1.set_xticks(YEARS_AXIS)
ax1.set_xticklabels([f"{y}년차" for y in YEARS_AXIS], fontsize=9.2)
ax1.set_ylabel("총 평가자산 (억 원)", fontsize=10.5, fontweight="bold")
ax1.set_title(f"① [{PRIMARY_NAME}] {H}년 자산 성장 시나리오 (몬테카를로 {SIM_CONFIG['N_SIMS']:,}회)",
              fontsize=13.2, fontweight="bold", color="#0F2922", loc="left", pad=26)
ax1.legend(loc="upper left", fontsize=8.8, frameon=True, facecolor="#F8FAFC", edgecolor="#CBD5E1")
ax1.grid(True, color="#E2E8F0", ls="--", lw=0.8)
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
fig1.tight_layout()
b1 = io.BytesIO(); fig1.savefig(b1, format="png", dpi=170, bbox_inches="tight", facecolor="white")
CHART1_B64 = base64.b64encode(b1.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig1)

# ------------------------------------------------------------------------------
# 차트 ② (풀폭 단독): 연도별 세후 월배당 스노우볼 & 금융소득종합과세(연 2,000만 원) 경보
# ------------------------------------------------------------------------------
fig2, (ax2l, ax2r) = plt.subplots(1, 2, figsize=(14.0, 5.8), facecolor="white")
bars_m = ax2l.bar(YEARS_AXIS, PRI_RES["mdiv_p50"], color="#0F766E", width=0.62, label="세후 월배당 (중앙값 50%)", zorder=3)
ax2l.plot(YEARS_AXIS, PRI_RES["mdiv_p10"], color="#D97706", ls="--", lw=1.8, marker=".", label="보수적 월배당 (하위 10%)", zorder=4)
for tgt, col in zip(SIM_CONFIG["TARGET_MONTHLY_DIVS_MAN"], ["#10B981", "#2563EB", "#7C3AED"]):
    if tgt <= max(PRI_RES["mdiv_p90"].max() * 1.15, 220):
        ax2l.axhline(tgt, color=col, ls=":", lw=1.3, zorder=2)
        ax2l.text(0.15, tgt + 3, f"목표 월 {tgt}만", fontsize=8.3, fontweight="bold", color=col)
for yr_i, v in enumerate(PRI_RES["mdiv_p50"]):
    if yr_i in (0, 3, 5, 8, y10, H):
        ax2l.text(yr_i, v + 4, f"{v:.0f}만", ha="center", va="bottom", fontsize=8.4, fontweight="bold", color="#0F2922")
ax2l.set_xticks(YEARS_AXIS)
ax2l.set_xticklabels([f"{y}년" for y in YEARS_AXIS], fontsize=8.4)
ax2l.set_ylabel("세후 월배당금 (만 원/월)", fontsize=10.0, fontweight="bold")
ax2l.set_title("②-A 연도별 세후 월배당 스노우볼 추이", fontsize=11.6, fontweight="bold", color="#0F2922", loc="left", pad=10)
ax2l.legend(loc="upper left", fontsize=8.4, frameon=True, facecolor="#F8FAFC")
ax2l.grid(True, axis="y", color="#E2E8F0", ls="--", lw=0.8)
ax2l.spines["top"].set_visible(False); ax2l.spines["right"].set_visible(False)

# 우측: 일반계좌 100% vs 절세배치 시 연간 과세 대상 배당소득 vs 2,000만 원 경보선
tax_0 = TAX_COMP["일반계좌 100% (절세 0%)"]
tax_70 = TAX_COMP["절세 극대화 (70%)"]
ax2r.plot(YEARS_AXIS, tax_0["norm_div_p50"], color="#DC2626", lw=2.3, marker="s", ms=4.5, label="일반계좌 100% 과세 배당소득")
ax2r.plot(YEARS_AXIS, PRI_RES["norm_div_p50"], color="#0F766E", lw=2.6, marker="o", ms=5, label=f"현행 절세배치({SIM_CONFIG['TAX_ADV_RATIO']:.0%}) 과세 배당소득")
ax2r.plot(YEARS_AXIS, tax_70["norm_div_p50"], color="#2563EB", lw=2.0, ls="--", marker="^", ms=4.5, label="절세 극대화(70%) 과세 배당소득")
ax2r.axhline(SIM_CONFIG["COMP_TAX_THRESHOLD_MAN"], color="#B91C1C", ls=":", lw=2.0)
ax2r.axhspan(SIM_CONFIG["COMP_TAX_THRESHOLD_MAN"], max(tax_0["norm_div_p50"].max() * 1.15, 2500), color="#FEE2E2", alpha=0.45)
ax2r.text(0.2, SIM_CONFIG["COMP_TAX_THRESHOLD_MAN"] + 60, "금융소득종합과세 경보선 (연 2,000만 원)", fontsize=8.8, fontweight="bold", color="#991B1B")
ax2r.set_xticks(YEARS_AXIS)
ax2r.set_xticklabels([f"{y}년" for y in YEARS_AXIS], fontsize=8.4)
ax2r.set_ylabel("일반계좌 연간 배당소득 (만 원/년)", fontsize=10.0, fontweight="bold")
ax2r.set_title("②-B 금융소득종합과세(2,000만 원) 도달 시점 비교", fontsize=11.6, fontweight="bold", color="#0F2922", loc="left", pad=10)
ax2r.legend(loc="upper left", fontsize=8.2, frameon=True, facecolor="#F8FAFC")
ax2r.grid(True, color="#E2E8F0", ls="--", lw=0.8)
ax2r.spines["top"].set_visible(False); ax2r.spines["right"].set_visible(False)

fig2.tight_layout()
b2 = io.BytesIO(); fig2.savefig(b2, format="png", dpi=170, bbox_inches="tight", facecolor="white")
CHART2_B64 = base64.b64encode(b2.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig2)

# ------------------------------------------------------------------------------
# 차트 ③ (풀폭 단독): 배분 전략별 10년차 성과 비교 + 절세계좌 10년 누적 세금 절감액
# ------------------------------------------------------------------------------
fig3, (ax3l, ax3r) = plt.subplots(1, 2, figsize=(14.0, 5.6), facecolor="white")
s_names = list(STRAT_RES.keys())
s_short = [s.split(" (")[0] + "\n(" + s.split(" (")[1] for s in s_names]
x_s = np.arange(len(s_names))
p10_vals = [STRAT_RES[s]["asset_p10"][y10]/10000 for s in s_names]
p50_vals = [STRAT_RES[s]["asset_p50"][y10]/10000 for s in s_names]
p90_vals = [STRAT_RES[s]["asset_p90"][y10]/10000 for s in s_names]
w_b = 0.25
ax3l.bar(x_s - w_b, p10_vals, width=w_b, color="#F59E0B", label="보수 (하위 10%)")
ax3l.bar(x_s,       p50_vals, width=w_b, color="#0F766E", label="기본 (중앙 50%)")
ax3l.bar(x_s + w_b, p90_vals, width=w_b, color="#10B981", label="낙관 (상위 10%)")
for xi, (v1, v5, v9) in enumerate(zip(p10_vals, p50_vals, p90_vals)):
    ax3l.text(xi - w_b, v1 + 0.12, f"{v1:.1f}억", ha="center", fontsize=8.3, fontweight="bold", color="#B45309")
    ax3l.text(xi,       v5 + 0.12, f"{v5:.1f}억", ha="center", fontsize=8.8, fontweight="bold", color="#0F2922")
    ax3l.text(xi + w_b, v9 + 0.12, f"{v9:.1f}억", ha="center", fontsize=8.3, fontweight="bold", color="#047857")
ax3l.set_xticks(x_s); ax3l.set_xticklabels(s_short, fontsize=9.0, fontweight="bold")
ax3l.set_ylabel(f"{y10}년차 평가자산 (억 원)", fontsize=10.0, fontweight="bold")
ax3l.set_title(f"③-A 배분 전략별 {y10}년차 자산 분포 비교", fontsize=11.6, fontweight="bold", color="#0F2922", loc="left", pad=10)
ax3l.set_ylim(0, max(p90_vals) * 1.26)
ax3l.legend(loc="upper center", ncol=3, fontsize=8.1, frameon=True, facecolor="#F8FAFC", edgecolor="#CBD5E1")
ax3l.grid(True, axis="y", color="#E2E8F0", ls="--", lw=0.8)
ax3l.spines["top"].set_visible(False); ax3l.spines["right"].set_visible(False)

# 우측: 절세계좌 비중별 10년 누적 세금 비교
t_names = list(TAX_COMP.keys())
t_vals = [TAX_COMP[k]["cum_tax_p50"][y10 - 1] for k in t_names]
t_cols = ["#DC2626", "#0F766E", "#2563EB"]
xt = np.arange(len(t_names))
ax3r.bar(xt, t_vals, color=t_cols, width=0.52)
base_t = t_vals[0]
for xi, tv in enumerate(t_vals):
    saved = base_t - tv
    lbl = f"{tv:,.0f}만 원" + (f"\n(▼ {saved:,.0f}만 절세)" if saved > 10 else "\n(기준)")
    ax3r.text(xi, tv + max(t_vals) * 0.03, lbl, ha="center", va="bottom", fontsize=9.0, fontweight="bold", color=t_cols[xi])
ax3r.set_xticks(xt); ax3r.set_xticklabels([t.replace(" (", "\n(") for t in t_names], fontsize=9.0, fontweight="bold")
ax3r.set_ylabel(f"{y10}년 누적 배당세금 (만 원)", fontsize=10.0, fontweight="bold")
ax3r.set_title(f"③-B ISA·연금 절세계좌 배치 시 {y10}년 누적 세금 절감", fontsize=11.6, fontweight="bold", color="#0F2922", loc="left", pad=10)
ax3r.set_ylim(0, max(t_vals) * 1.25)
ax3r.grid(True, axis="y", color="#E2E8F0", ls="--", lw=0.8)
ax3r.spines["top"].set_visible(False); ax3r.spines["right"].set_visible(False)

fig3.tight_layout()
b3 = io.BytesIO(); fig3.savefig(b3, format="png", dpi=170, bbox_inches="tight", facecolor="white")
CHART3_B64 = base64.b64encode(b3.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig3)

# ------------------------------------------------------------------------------
# HTML / PDF 5페이지 고가독성 리포트 조립
# ------------------------------------------------------------------------------
def render_clean_tbl(df: pd.DataFrame, highlight_row: int = 0) -> str:
    ths = "".join(f"<th>{escape(str(c))}</th>" for c in df.columns)
    rows = []
    for idx_r, (_, r) in enumerate(df.iterrows()):
        cls = "row-hi" if idx_r == highlight_row else ""
        tds = "".join(f'<td class="{"col-left" if i==0 else ""}">{escape(str(r[c]))}</td>' for i, c in enumerate(df.columns))
        rows.append(f"<tr class='{cls}'>{tds}</tr>")
    return f'<table class="tbl"><thead><tr>{ths}</tr></thead><tbody>{"".join(rows)}</tbody></table>'

tax_saved_10y = TAX_COMP["일반계좌 100% (절세 0%)"]["cum_tax_p50"][y10 - 1] - PRI_RES["cum_tax_p50"][y10 - 1]
comp_hit_norm = np.where(TAX_COMP["일반계좌 100% (절세 0%)"]["norm_div_p50"] >= SIM_CONFIG["COMP_TAX_THRESHOLD_MAN"])[0]
comp_hit_cur  = np.where(PRI_RES["norm_div_p50"] >= SIM_CONFIG["COMP_TAX_THRESHOLD_MAN"])[0]
comp_norm_txt = f"{int(comp_hit_norm[0])}년차 돌파" if len(comp_hit_norm) else f"{H}년 내 미돌파"
comp_cur_txt  = f"{int(comp_hit_cur[0])}년차 돌파" if len(comp_hit_cur) else f"{H}년 내 안전"

tbl_strat_html = render_clean_tbl(SUMMARY_SIM_DF, highlight_row=0)
tbl_goal_html  = render_clean_tbl(GOAL_DF, highlight_row=1 if len(GOAL_DF) > 1 else 0)

html_report = f"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><title>10년 배당 현금흐름 · 절세 시뮬레이션 리포트 ({as_of_str})</title>
<style>
  @page {{ size: A4 landscape; margin: 9mm 10mm; }}
  body {{ margin:0; padding:10px; background:#F8FAFC; color:#0F172A; font-family:'NanumGothic',sans-serif; font-size:10.3px; line-height:1.5; }}
  .wrap {{ max-width:1120px; margin:0 auto; }}
  .head {{ background:linear-gradient(135deg,#0F2922,#134E4A); color:#fff; padding:15px 20px; border-radius:10px; margin-bottom:10px; border-bottom:4px solid #D4A34A; }}
  .head h1 {{ margin:0 0 4px; font-size:17.5px; }}
  .head .sub {{ font-size:9.8px; opacity:0.92; }}
  .kpi-row {{ display:grid; grid-template-columns:repeat(4,1fr); gap:10px; margin-bottom:10px; }}
  .kpi {{ background:#fff; border:1px solid #E2E8F0; border-top:3px solid #0F766E; border-radius:8px; padding:9px 11px; }}
  .kpi .k-lbl {{ font-size:8.8px; color:#64748B; font-weight:700; }}
  .kpi .k-val {{ font-size:15.5px; font-weight:800; color:#0F2922; margin:2px 0; }}
  .kpi .k-sub {{ font-size:8.4px; color:#475569; }}
  .guide-box {{ background:#F0FDF9; border:1px solid #99F6E4; border-left:4px solid #0F766E; border-radius:8px; padding:9px 13px; margin-bottom:10px; font-size:9.3px; }}
  .guide-grid {{ display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin-top:5px; }}
  .card {{ background:#fff; border:1px solid #E2E8F0; border-radius:10px; padding:11px 14px; margin-bottom:10px; break-inside:avoid; }}
  .card h2 {{ margin:0 0 8px; font-size:12.6px; color:#0F2922; border-bottom:1.5px solid #E2E8F0; padding-bottom:5px; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:9.4px; }}
  table.tbl th {{ background:#0F2922; color:#fff; padding:6.5px 7px; text-align:right; font-weight:700; }}
  table.tbl td {{ padding:6.5px 7px; border-bottom:1px solid #E2E8F0; text-align:right; }}
  table.tbl tr.row-hi {{ background:#ECFDF5; font-weight:700; }}
  table.tbl td.col-left, table.tbl th:first-child {{ text-align:left; font-weight:700; }}
  .note-box {{ background:#FFFBEB; border:1px solid #FDE68A; border-left:4px solid #D97706; border-radius:7px; padding:8px 11px; margin-top:8px; font-size:9.1px; color:#78350F; line-height:1.55; }}
  .chart-tip {{ background:#F8FAFC; border:1px solid #E2E8F0; border-left:4px solid #0F766E; border-radius:6px; padding:7px 10px; margin-top:8px; font-size:9.2px; color:#334155; }}
  img.chart {{ width:100%; height:auto; display:block; }}
  .page-break {{ page-break-before:always; break-before:page; }}
</style></head><body><div class="wrap">

  <!-- PAGE 1: 핵심 KPI + 30초 가이드 + 전략별 비교표 -->
  <div class="head">
    <h1>10년 배당 현금흐름 · 절세계좌 · 종합과세 시뮬레이션 리포트 ({as_of_str})</h1>
    <div class="sub">초기자산 {SIM_CONFIG['INIT_ASSET_MAN']:,}만 원 · 월 적립 {SIM_CONFIG['MONTHLY_CONTRIB_MAN']:,}만 원 · 절세계좌 비중 {SIM_CONFIG['TAX_ADV_RATIO']:.0%} · 상관관계 몬테카를로 {SIM_CONFIG['N_SIMS']:,}회 ({CALIB_SOURCE})</div>
  </div>

  <div class="kpi-row">
    <div class="kpi">
      <div class="k-lbl">{y10}년차 예상 총자산 (중앙값)</div>
      <div class="k-val">{PRI_RES['asset_p50'][y10]/10000:.2f}억 원</div>
      <div class="k-sub">보수(하위10%) {PRI_RES['asset_p10'][y10]/10000:.2f}억 ~ 낙관 {PRI_RES['asset_p90'][y10]/10000:.2f}억</div>
    </div>
    <div class="kpi">
      <div class="k-lbl">{y10}년차 세후 월배당 (중앙값)</div>
      <div class="k-val">월 {PRI_RES['mdiv_p50'][y10]:.0f}만 원</div>
      <div class="k-sub">현재 월 {PRI_RES['mdiv_p50'][0]:.0f}만 원 → {y10}년 뒤 약 {PRI_RES['mdiv_p50'][y10]/max(PRI_RES['mdiv_p50'][0],1):.1f}배 성장</div>
    </div>
    <div class="kpi">
      <div class="k-lbl">절세계좌 배치 {y10}년 절세액</div>
      <div class="k-val">약 {tax_saved_10y:,.0f}만 원 절감</div>
      <div class="k-sub">일반계좌 100% 대비 절세({SIM_CONFIG['TAX_ADV_RATIO']:.0%}) 효과</div>
    </div>
    <div class="kpi">
      <div class="k-lbl">금융소득종합과세(2천만) 경보</div>
      <div class="k-val">{comp_cur_txt}</div>
      <div class="k-sub">일반계좌 100% 시: {comp_norm_txt}</div>
    </div>
  </div>

  <div class="guide-box">
    <b>📖 30초 만에 읽는 이 리포트 활용법 (단순 미래 예측이 아닌 의사결정 도구!)</b>
    <div class="guide-grid">
      <div><b>1) 보수(하위 10%) 숫자를 기준으로 계획하세요:</b><br>시장이 안 좋게 흘러가도 90% 확률로 지킬 수 있는 하한선입니다. 내 목표가 보수 시나리오에서도 버틸 수 있는지 확인하세요.</div>
      <div><b>2) 왜 ISA·연금계좌에 고배당주(A층)를 먼저 채우나요?</b><br>일반계좌 배당이 연 2,000만 원을 넘으면 종합소득세+건보료 부담이 커집니다. 고배당 A층을 절세계좌에 넣을수록 돌파 시점이 수년 뒤로 밀립니다.</div>
      <div><b>3) 월 적립금 조절 레버로 활용하세요:</b><br>아래 「목표 월배당 달성표」에서 월 적립금을 늘리거나 배분안을 바꿀 때 은퇴·현금흐름 목표가 몇 년 앞당겨지는지 바로 확인할 수 있습니다.</div>
    </div>
  </div>

  <div class="card">
    <h2>01. 배분 전략별 {y10}년차 자산 · 세후 월배당 · 세금 한눈에 비교 (연녹색 = 현행 기준)</h2>
    {tbl_strat_html}
  </div>

  <!-- PAGE 2: 월 적립금별 목표 월배당 도달표 + 실전 절세 배치 가이드 -->
  <div class="card page-break">
    <h2>02. 월 적립금별 목표 세후 월배당(100만 · 200만 · 300만 원) 달성 예상 연도</h2>
    {tbl_goal_html}
    <div class="note-box">
      <b>💡 실전 절세 계좌 배치 우선순위 가이드 (같은 돈으로 세후 배당 극대화하기)</b><br>
      <b>① 1순위 — 고배당 국내주(A층, 배당률 ~4.8%):</b> 배당소득 비중이 가장 크므로 <b>ISA(비과세·9.9% 분리과세)</b> 및 <b>절세계좌</b>에 최우선 배치하세요.<br>
      <b>② 2순위 — 미국 지수 ETF(C층, 배당률 ~1.3%):</b> 국내 상장 미국 ETF는 매매차익에도 배당소득세(15.4%)가 붙으므로 <b>연금저축·IRP</b>에 넣으면 과세이연 복리 효과가 큽니다.<br>
      <b>③ 3순위 — 일반계좌 활용:</b> 국내 개별주 매매차익은 비과세(대주주 제외)이므로, 절세계좌 한도가 꽉 차면 <b>배당률이 낮은 성장형·반도체(B층)나 저배당 개별주</b>를 일반계좌에 두는 것이 세금 효율이 가장 높습니다.
    </div>
  </div>

  <!-- PAGE 3: 차트 ① 자산 성장 팬 차트 -->
  <div class="card page-break">
    <h2>03. [차트 ①] {H}년 총 평가자산 성장 시나리오 (5,000회 몬테카를로)</h2>
    <img class="chart" src="data:image/png;base64,{CHART1_B64}">
    <div class="chart-tip">
      <b>💡 그래프 읽는 법:</b> 진한 검정 실선이 <b>가장 가능성 높은 기본(중앙값) 경로</b>이며, 회색 점선은 배당·수익 없이 내가 모은 순수 원금입니다.
      노란 점선(보수 하위 10%)은 10번 중 1번 꼴의 불운한 장세를 가정한 하한선으로, 장기 적립과 배당 재투자가 결합되면 하한선도 시간이 갈수록 원금을 웃돌게 됩니다.
    </div>
  </div>

  <!-- PAGE 4: 차트 ② 월배당 스노우볼 + 종소세 2,000만 원 경보 -->
  <div class="card page-break">
    <h2>04. [차트 ②] 세후 월배당 스노우볼 &amp; 금융소득종합과세(연 2,000만 원) 경보 시점</h2>
    <img class="chart" src="data:image/png;base64,{CHART2_B64}">
    <div class="chart-tip">
      <b>💡 그래프 읽는 법:</b> 왼쪽 막대는 매월 통장에 꽂히는 <b>세후 월배당금</b>이 복리로 불어나는 모습입니다.
      오른쪽 그래프는 <b>일반계좌에서 나오는 연간 배당소득</b>이 빨간 점선(연 2,000만 원 종합과세선)을 언제 넘는지 보여줍니다. 절세계좌 비중을 높일수록(초록·파랑 선) 빨간 위험 구역 진입이 수년 이상 늦춰집니다.
    </div>
  </div>

  <!-- PAGE 5: 차트 ③ 배분 전략별 비교 + 절세액 -->
  <div class="card page-break">
    <h2>05. [차트 ③] 배분 전략별 {y10}년차 자산 분포 &amp; 절세계좌 누적 세금 절감액</h2>
    <img class="chart" src="data:image/png;base64,{CHART3_B64}">
    <div class="chart-tip">
      <b>💡 그래프 읽는 법:</b> 왼쪽은 3가지 배분안의 보수(주황)·기본(진녹)·낙관(연녹) {y10}년차 자산 비교입니다. C층(미국)을 섞은 안 ①·②가 국내 단독(안 ③) 대비 상방과 분산 방어력을 동시에 개선합니다.
      오른쪽은 같은 수익률·같은 종목을 들고 있어도 <b>계좌 배치(ISA·연금)만으로 아끼는 10년 누적 세금</b>을 보여줍니다.
    </div>
  </div>

</div></body></html>"""

r_html = f"cashflow_sim_report_{run_stamp}.html"
r_pdf  = f"cashflow_sim_report_{run_stamp}.pdf"
Path(r_html).write_text(html_report, encoding="utf-8")
pdf_ok = export_html_to_pdf(html_report, r_pdf, [CHART1_B64, CHART2_B64, CHART3_B64])
print(f"[리포트 완료] HTML={Path(r_html).stat().st_size/1024:.1f} KB | PDF={'성공' if pdf_ok else '건너뜀'}")

# Google Drive 자동 저장 (+ Cell 2 미연결 시 자동 재시도 & 실패 시 브라우저 다운로드)
if IN_COLAB and SIM_CONFIG.get("USE_GOOGLE_DRIVE", True) and DRIVE_REPORT_DIR is None:
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive", force_remount=True)
        DRIVE_ROOT = Path(SIM_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print(f"[Drive 미연결] 로컬(/content/{r_pdf if pdf_ok else r_html}) 저장됨 ({type(e).__name__}: {e})")
        try:
            from google.colab import files
            files.download(r_pdf if pdf_ok else r_html)
        except Exception:
            pass

if DRIVE_REPORT_DIR is not None:
    try:
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        (DRIVE_REPORT_DIR / r_html).write_text(html_report, encoding="utf-8")
        (DRIVE_REPORT_DIR / "cashflow_sim_report_latest.html").write_text(html_report, encoding="utf-8")
        if pdf_ok and Path(r_pdf).exists():
            pb = Path(r_pdf).read_bytes()
            (DRIVE_REPORT_DIR / r_pdf).write_bytes(pb)
            (DRIVE_REPORT_DIR / "cashflow_sim_report_latest.pdf").write_bytes(pb)
        SUMMARY_SIM_DF.to_csv(DRIVE_REPORT_DIR / f"cashflow_summary_{run_stamp}.csv", index=False, encoding="utf-8-sig")
        SUMMARY_SIM_DF.to_csv(DRIVE_REPORT_DIR / "cashflow_summary_latest.csv", index=False, encoding="utf-8-sig")
        print(f"[Drive 저장 검증 완료] {DRIVE_REPORT_DIR / 'cashflow_sim_report_latest.pdf'}")
    except Exception as e:
        print(f"[주의] Drive 저장 실패({type(e).__name__}): {e}")